In [1]:
import numpy as np
import tensorflow as tf

In [2]:
# Two vectors examples
# Input data

v1 = np.array([1, 2, 3], dtype=float)
v2 = np.array([1, 2, 3.5], dtype=float) # notice the 3rd element is offset by 0.5

print("-- Inputs --")
print(f"v1 :\n{v1}\n")
print(f"v2 :\n{v2}\n")

# Similarity scores
def cosine_similarity (v1, v2):
    numerator = tf.math.reduce_sum(v1*v2) #takes the dot product between v1 and v2, Equivalent to np.dot (v1, v2)
    denominator = tf.math.sqrt(tf.math.reduce_sum(v1*v1) * tf.math.reduce_sum(v2*v2))
    return numerator / denominator

print(" --Outputs-- ")
print("cosine similarity:", cosine_similarity(v1,v2).numpy)

-- Inputs --
v1 :
[1. 2. 3.]

v2 :
[1.  2.  3.5]

 --Outputs-- 
cosine similarity: <bound method _EagerTensorBase.numpy of <tf.Tensor: shape=(), dtype=float64, numpy=0.9974086507360697>>


In [3]:
# Two batches of vectors examples
# Input data

v1_1 = np.array([1.0, 2.0, 3.0])
v1_2 = np.array([9.0,8.0, 7.0])
v1_3 = np.array([-1.0, -4.0, -2.0])
v1_4 = np.array([1.0, -7.0, -2.0])
v1= np.vstack([v1_1,v1_2 , v1_3, v1_4])

v2_1 = v1_1 + np.random.normal(0, 2, 3) # add some noise to create approximate duplicate
v2_2 = v1_2 + np.random.normal(0, 2, 3) 
v2_3 = v1_3 + np.random.normal(0, 2, 3) 
v2_4 = v1_4 + np.random.normal(0, 2, 3) 
v2 = np.vstack([v2_1,v2_2 , v2_3, v2_4])

print("-- Inputs --")
print(f"v1 :\n{v1}\n")
print(f"v2 :\n{v2}\n")

# Batch sizes must match
b = len(v1)
print (f"Batch sizes match : {b == len(v2)}")

# Similarity scores

# Option 1: nested loops and the cosine similarity function
sim_1 = np.zeros([b, b]) # empty array to take similarity scores

# Loop
for row in range (0, sim_1.shape[0]):
    for col in range (0, sim_1.shape[1]):
        sim_1[row, col] = cosine_similarity(v2[row], v1[col]).numpy()

print(" --Outputs-- ")
print("Option 1 : loop")
print(sim_1)

-- Inputs --
v1 :
[[ 1.  2.  3.]
 [ 9.  8.  7.]
 [-1. -4. -2.]
 [ 1. -7. -2.]]

v2 :
[[ 3.81600566  1.66958695  5.78348186]
 [ 9.70146109  8.755635   10.64125688]
 [-3.33094732 -3.87625966 -2.71436817]
 [-1.03084614 -7.79059221 -3.37960866]]

Batch sizes match : True
 --Outputs-- 
Option 1 : loop
[[ 0.91892086  0.88832275 -0.67545766 -0.37113553]
 [ 0.93782182  0.98770986 -0.85468459 -0.58841478]
 [-0.8879544  -0.99239193  0.91499458  0.68739861]
 [-0.83576508 -0.79950005  0.99365468  0.95865189]]


In [4]:
# Option 2 : vector normalization and dot product
def norm(x): 
    return tf.math.l2_normalize(x, axis = 1) # use tensorflow built in normalization

sim_2 = tf.linalg.matmul(norm(v2), norm(v1), transpose_b=True)

print(" --Outputs-- ")
print("Option 2 : vector normalization and dot product")
print(sim_2, "\n")

# Check
print (f"Outputs are the same : {np.allclose(sim_1, sim_2)}")



 --Outputs-- 
Option 2 : vector normalization and dot product
tf.Tensor(
[[ 0.91892086  0.88832275 -0.67545766 -0.37113553]
 [ 0.93782182  0.98770986 -0.85468459 -0.58841478]
 [-0.8879544  -0.99239193  0.91499458  0.68739861]
 [-0.83576508 -0.79950005  0.99365468  0.95865189]], shape=(4, 4), dtype=float64) 

Outputs are the same : True


In [5]:
# Hardcoded matrix of similarity scores using numpy
sim_hardcoded = np.array(
    [
        [0.9, -0.8, 0.3, -0.5],
        [-0.4, 0.5, 0.1, -0.1],
        [0.3, 0.1, -0.4, -0.8],
        [-0.5, -0.2, -0.7, 0.5],
    ]
)

sim = sim_hardcoded

# Batch size
b = sim.shape[0]

print(" --Inputs--")
print(f"sim:")
print(sim)
print(f"shape: {sim.shape}\n")

# Positive
# All the s(A,P) values : similarities from duplicate question pairs (aka positive)
# These are along the diagonals
sim_ap = np.diag(sim)
print("sim_ap:")
print(np.diag(sim_ap))

# Negative
# All the s(A,P) values : similarities from duplicate question pairs (aka Negative)
# These are in the off diagonals
sim_an = sim - np.diag(sim_ap)
print("\nsim_an:")
print(np.diag(sim_an))

print(" \n--Outputs--")
# Mean negative
# Average of the s(A,N) values for each row
mean_neg = np.sum(sim_an, axis=1, keepdims=True) / (b - 1)
print("\nmean_neg:")
print(mean_neg)

# Closest negative
# Max s(A,N) that is <= s(A,P) for each row
mask_1 = np.identity(b) == 1 # Mask to exclude the diagonal
mask_2 = sim_an > sim_ap.reshape(b, 1) # Mask to exclude sim_an > sim_ap
mask = mask_1 | mask_2
sim_an_masked = np.copy(sim_an)
sim_an_masked[mask] = -2

closest_neg = np.max(sim_an_masked, axis=1, keepdims= True)
print("\nclosest_neg : ")
print(closest_neg)


 --Inputs--
sim:
[[ 0.9 -0.8  0.3 -0.5]
 [-0.4  0.5  0.1 -0.1]
 [ 0.3  0.1 -0.4 -0.8]
 [-0.5 -0.2 -0.7  0.5]]
shape: (4, 4)

sim_ap:
[[ 0.9  0.   0.   0. ]
 [ 0.   0.5  0.   0. ]
 [ 0.   0.  -0.4  0. ]
 [ 0.   0.   0.   0.5]]

sim_an:
[0. 0. 0. 0.]
 
--Outputs--

mean_neg:
[[-0.33333333]
 [-0.13333333]
 [-0.13333333]
 [-0.46666667]]

closest_neg : 
[[ 0.3]
 [ 0.1]
 [-0.8]
 [-0.2]]


In [9]:
# Hardcoded matrix of similarity scores using Tensorflow
sim_hardcoded = np.array(
    [
        [0.9, -0.8, 0.3, -0.5],
        [-0.4, 0.5, 0.1, -0.1],
        [0.3, 0.1, -0.4, -0.8],
        [-0.5, -0.2, -0.7, 0.5],
    ]
)

sim = sim_hardcoded

# Batch size
b = sim.shape[0]

print(" --Inputs--")
print(f"sim:")
print(sim)
print(f"shape: {sim.shape}\n")

# Positive
# All the s(A,P) values : similarities from duplicate question pairs (aka positive)
# These are along the diagonals
sim_ap = tf.linalg.diag_part(sim)
print("sim_ap:")
# tf.linalg.diag makes a diagonal matrix given an array
print(tf.linalg.diag(sim_ap),"/n" )

# Negative
# All the s(A,P) values : similarities from duplicate question pairs (aka Negative)
# These are in the off diagonals
sim_an = sim - tf.linalg.diag(sim_ap)
print("sim_an:")
print(sim_an, "\n")

print(" --Outputs--")
# Mean negative
# Average of the s(A,N) values for each row
mean_neg = tf.math.reduce_sum(sim_an, axis=1) / (b - 1)

# Closest negative
# Max s(A,N) that is <= s(A,P) for each row
mask_1 = tf.eye(b) == 1 # Mask to exclude the diagonal
mask_2 = sim_an > tf.expand_dims(sim_ap, 1) # Mask to exclude sim_an > sim_ap
mask = tf.cast(mask_1 | mask_2, tf.float64)
sim_an_masked = sim_an - 2.0*mask

closest_neg = tf.math.reduce_max(sim_an_masked, axis=1)
print("closest_neg : ")
print(closest_neg, "\n")


 --Inputs--
sim:
[[ 0.9 -0.8  0.3 -0.5]
 [-0.4  0.5  0.1 -0.1]
 [ 0.3  0.1 -0.4 -0.8]
 [-0.5 -0.2 -0.7  0.5]]
shape: (4, 4)

sim_ap:
tf.Tensor(
[[ 0.9  0.   0.   0. ]
 [ 0.   0.5  0.   0. ]
 [ 0.   0.  -0.4  0. ]
 [ 0.   0.   0.   0.5]], shape=(4, 4), dtype=float64) /n
sim_an:
tf.Tensor(
[[ 0.  -0.8  0.3 -0.5]
 [-0.4  0.   0.1 -0.1]
 [ 0.3  0.1  0.  -0.8]
 [-0.5 -0.2 -0.7  0. ]], shape=(4, 4), dtype=float64) 

 --Outputs--
closest_neg : 
tf.Tensor([ 0.3  0.1 -0.8 -0.2], shape=(4,), dtype=float64) 



In [11]:
# Alpha margin
alpha = 0.25

# Modified triplets loss
# Loss 1
l_1 = tf.maximum(mean_neg - sim_ap + alpha, 0)
print (f"Loss 1 :{l_1} \n")

# Loss 2
l_2 = tf.maximum(closest_neg - sim_ap + alpha, 0)
print (f"Loss 2 :{l_2} \n")

# Loss full <
l_full = l_1 + l_2

# Cost
cost = tf.math.reduce_sum(l_full)

print(" --Outputs-- ")
print ("Loss full :")
print (l_full, "\n")
print ("Cost :", "{:.3f}".format(cost))

Loss 1 :[0.         0.         0.51666667 0.        ] 

Loss 2 :[0. 0. 0. 0.] 

 --Outputs-- 
Loss full :
tf.Tensor([0.         0.         0.51666667 0.        ], shape=(4,), dtype=float64) 

Cost : 0.517
